# google/gemma-4-12B-it 회의 안건 생성 & 평가 (RunPod)

> **모델**: `google/gemma-4-12B-it`  
> **권장 VRAM**: 4bit 양자화 기준 약 8~10GB

In [1]:
# 패키지 설치 (실행 후 커널 재시작 -> 다음 셀부터 다시 실행)
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

In [2]:
# HuggingFace 로그인 (gemma는 gated 모델 -> 토큰 필수)
# huggingface.co -> Settings -> Access Tokens 에서 발급
# huggingface.co/google/gemma-4-12B-it 에서 사용 동의(Accept)도 필요
import os
os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"  # 여기에 토큰 입력

In [3]:
import time
import gc
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"transformers: {transformers.__version__}")
print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    print(f"VRAM 전체: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

transformers: 5.10.2
GPU 사용 가능: True
GPU 이름: NVIDIA A40
VRAM 전체: 44.4 GB


In [4]:
MODEL_ID = "google/gemma-4-12B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"로드 중: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM 사용: {used:.2f} GB / {total:.1f} GB")
print("로드 완료")

로드 중: google/gemma-4-12B-it


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

VRAM 사용: 7.15 GB / 44.4 GB
로드 완료


In [5]:
def generate(messages, max_new_tokens=600):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

print("generate 함수 정의 완료")

generate 함수 정의 완료


## 회의 데이터

In [6]:
previous_meeting = """
[회의록] 2024년 11월 18일 (월) 스프린트 회고 회의
참석자: 김대표, 이CTO, 박PM, 최백엔드, 정프론트, 한디자이너

논의 내용
1. 스프린트 #12 회고
   - 회원가입/로그인 기능 완료, QA 통과
   - 대시보드 UI 3일 지연 (디자인 시안 수정 반복)
   - API 응답속도 800ms 초과, 최적화 필요
2. 투자 업데이트
   - 시리즈A 투자사 2곳 미팅 완료
   - MAU 지표와 리텐션 데이터 추가 요청
3. 팀 운영
   - 백엔드 개발자 1명 채용 결정
   - 온보딩 프로세스 문서화 미비

액션 아이템
- [이CTO] API 병목 구간 분석 보고서
- [박PM] 투자사 요청 지표 항목 정리
- [한디자이너] 대시보드 UI 확정 시안 공유
"""


unresolved_tasks = """
[미해결 태스크]
1. [높음] API 응답속도 최적화 - 담당: 최백엔드 (목표: 300ms)
2. [높음] 투자사 지표 대시보드 - 담당: 박PM+정프론트 (마감: 12월 5일)
3. [중간] 온보딩 플로우 개선 - 담당: 한디자이너+정프론트
4. [중간] 백엔드 개발자 채용 - 담당: 김대표+이CTO
5. [낮음] 고객 문의 대응 개선 - 담당: 박PM
"""

meeting_info = """
회의명: 12월 첫째 주 전체 팀 스탠드업
일시: 2024년 12월 2일 (월) 오전 10시
참석자: 전체 팀원 6명 / 예상 소요시간: 1시간
"""

print("회의 데이터 로드 완료")

회의 데이터 로드 완료


## 회의 안건 생성 (생성 시간 측정)

In [7]:
generation_messages = [
    {
        "role": "user",
        "content": (
            "당신은 회의 기초안건을 작성하는 전문 비서입니다. 주어진 자료를 분석하여 실용적이고 구체적인 회의 안건을 작성하세요.\n\n"
            "아래 자료를 바탕으로 회의 기초안건을 작성해주세요.\n\n"
            f"[회의 정보]\n{meeting_info}\n\n"
            f"[이전 회의록]\n{previous_meeting}\n\n"
            f"[미해결 태스크]\n{unresolved_tasks}\n\n"
            "위 자료를 종합하여 이번 회의에서 반드시 다뤄야 할 안건을 작성해주세요. "
            "각 안건에는 논의 목적과 주요 논의 포인트를 포함해주세요."
        )
    }
]

print("기초안건 생성 중...")
start = time.time()
generated_agenda = generate(generation_messages, max_new_tokens=600)
elapsed = time.time() - start

token_count = len(tokenizer.encode(generated_agenda))

print("\n" + "="*60)
print("[생성된 기초안건]")
print("="*60)
print(generated_agenda)
print("\n" + "="*60)
print(f"모델         : {MODEL_ID}")
print(f"생성 시간    : {elapsed:.2f}초")
print(f"생성 토큰 수 : {token_count}")
print(f"토큰/초      : {token_count / elapsed:.1f}")
print("="*60)

기초안건 생성 중...

[생성된 기초안건]
안녕하십니까, 회의 기초안건 작성을 담당하는 비서입니다.

제공해주신 이전 회의록, 월간 서비스 지표, 그리고 미해결 태스크를 종합 분석하여 **[12월 첫째 주 전체 팀 스탠드업]**을 위한 회의 기초안건을 다음과 같이 제안합니다.

---

# [회의 기초안건] 12월 첫째 주 전체 팀 스탠드업

**■ 회의 개요**
*   **일시:** 2024년 12월 2일 (월) 오전 10:00 ~ 11:00 (60분)
*   **참석자:** 전체 팀원 6명 (김대표, 이CTO, 박PM, 최백엔드, 정프론트, 한디자이너)
*   **회의 목적:** 11월 성과 및 지표 리뷰, 핵심 미해결 태스크(Backlog) 진척도 점검 및 12월 초 핵심 마일스톤 확정

---

### [의제 1] 11월 성과 리뷰 및 지표 분석 (15분)
> **목적:** 전월 성과를 객관적으로 파악하고, 개선이 시급한 지표에 대한 대응 방안 논의

*   **주요 논의 포인트:**
    *   **성장 지표:** MAU 18% 성장 및 리텐션 현황 공유.
    *   **위기 지표 분석:** 3일 내 이탈률(41%)이 높은 원인 분석 및 '온보딩 플로우 개선' 과제 우선순위 재확인.
    *   **기술 지표:** 모바일 크래시율(1.8%) 및 고객 문의 응답 지연(14시간) 문제 해결을 위한 부서별 협조 방안.
    *   **경쟁 상황:** 12월 출시 예정인 경쟁사 A의 유사 기능 대응 전략 초동 논의.

### [의제 2] 핵심 미해결 태스크(High Priority) 진척도 점검 (20분)
> **목적:** 12월 초 마감 예정인 핵심 과제의 병목 현상 파악 및 리소스 조정

*   **주요 논의 포인트:**
    *   **API 최적화:** [최백엔드] 현재 응답속도 개선 현황 및 목표(300ms) 달성을 위한 기술적 장애 요소 공유.
    *   **투자용 지표 대시보드:** [박PM, 정프론트] 12월 5일 마감 예정인 

## 회의 안건 평가

In [ ]:
rubric = """
[평가 루브릭] 각 항목을 아래 기준에 따라 1~5점으로 채점

1) 안건 관련성
- 1점: 생성된 안건이 회의 주제와 전혀 무관함
- 2점: 일부 관련 있으나 절반 이상 주제 이탈
- 3점: 주제와 관련 있으나 핵심 안건 일부 누락
- 4점: 대부분 주제에 부합하며 경미한 이탈만 존재
- 5점: 모든 안건이 회의 주제와 정확히 일치함

2) 안건 구체성
- 1점: '논의 예정' 등 모호한 표현만 존재, 실행 불가
- 2점: 항목명만 있고 세부 내용 없음
- 3점: 일부 항목은 구체적이나 절반 이상 모호함
- 4점: 대부분 구체적이며 일부 보완 필요
- 5점: 모든 항목이 실행 가능한 수준으로 명확함

3) 외부자료 반영도 (외부자료 = 이전 회의록, 미해결 태스크)
- 1점: 외부자료가 전혀 반영되지 않음
- 2점: 외부자료 언급은 있으나 내용 반영 미흡
- 3점: 외부자료 일부 반영, 핵심 내용 누락
- 4점: 외부자료 주요 내용 반영, 일부 세부사항 누락
- 5점: 외부자료 핵심 내용이 안건에 완전히 반영됨

4) 항목 완결성
- 1점: 안건 항목이 1개 이하이거나 대부분 누락
- 2점: 주요 안건 절반 이상 누락
- 3점: 주요 안건 포함되나 세부 항목 누락 다수
- 4점: 대부분의 안건 포함, 1~2개 누락
- 5점: 모든 필요 안건이 빠짐없이 도출됨

5) 한국어 품질
- 1점: 문장이 어색하거나 비문/오탈자 다수
- 2점: 의미 전달은 되나 어색한 표현 다수
- 3점: 전반적으로 자연스러우나 일부 어색한 표현
- 4점: 자연스러운 한국어, 경미한 어색함만 존재
- 5점: 완전히 자연스럽고 격식에 맞는 한국어
"""

evaluation_messages = [
    {
        "role": "user",
        "content": (
            "당신은 회의 안건의 품질을 평가하는 전문가입니다. 주어진 루브릭의 점수 기준을 엄격히 적용하여 객관적으로 채점하세요.\n\n"
            "아래 회의 안건을 5가지 항목으로 평가해주세요.\n"
            "반드시 각 항목의 1~5점 기준에 비추어 채점하고, 그 점수를 준 이유를 한 줄로 설명하세요.\n\n"
            f"{rubric}\n"
            "[평가 대상 입력 자료]\n"
            f"- 회의 정보:\n{meeting_info}\n"
            f"- 이전 회의록:\n{previous_meeting}\n"
            f"- 미해결 태스크:\n{unresolved_tasks}\n\n"
            f"[평가할 안건]\n{generated_agenda}\n\n"
            "출력 형식(반드시 이 형식으로):\n"
            "1. 안건 관련성: X/5 - (이유)\n"
            "2. 안건 구체성: X/5 - (이유)\n"
            "3. 외부자료 반영도: X/5 - (이유)\n"
            "4. 항목 완결성: X/5 - (이유)\n"
            "5. 한국어 품질: X/5 - (이유)\n"
            "총점: X/25\n"
            "종합 의견:"
        )
    }
]

print("평가 중...")
evaluation_result = generate(evaluation_messages, max_new_tokens=700)

print("\n" + "="*60)
print("[평가 결과]")
print("="*60)
print(evaluation_result)

평가 중...


In [ ]:
del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"모델 해제 완료 | 잔여 VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")